# 02 — Entrenamiento de modelos
A partir de la tabla analítica base construimos tres modelos que miran la mortalidad en Guatemala desde ángulos distintos: **cuánto exceso dejó la pandemia**, **qué factores explican las defunciones** y **cómo se reorganizó el perfil de causas por departamento**. Cada modelo guarda sus resultados en una tabla `pred_*` para poder visualizarlos después.

In [ ]:
# Databricks Runtime ML ya incluye scikit-learn, mlflow, joblib, matplotlib.
# Si estas en un cluster estandar (no ML), descomenta la siguiente linea:
# %pip install -q scikit-learn mlflow joblib


In [ ]:
# --- Verificacion de conexion S3 (mismo patron que notebooks/phase#1/s3.py:48) ---
import boto3
_c = boto3.client('s3', region_name='us-east-1',
    aws_access_key_id=dbutils.secrets.get('semis2_scope', 'AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=dbutils.secrets.get('semis2_scope', 'AWS_SECRET_ACCESS_KEY'))
_objs = _c.list_objects_v2(Bucket='mortality-analytics-semi2', Prefix='ml/').get('Contents', [])
print(f'S3 OK -> {len(_objs)} objetos bajo ml/ (si es 0, primero corré 01_build_abt)')
for o in _objs[:5]:
    print(f'   {o["Key"]}  ({o["Size"]} bytes)')


In [ ]:
import pandas as pd, numpy as np, os, joblib, mlflow, boto3
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, r2_score, silhouette_score

CAT = 'semi2.dm_mortality'
BUCKET = 'mortality-analytics-semi2'
S3_ML = 'ml'
S3_ABT = f'{S3_ML}/abt'
S3_MODELS = f'{S3_ML}/models'
S3_PRED = f'{S3_ML}/pred'
S3_FIG = f'{S3_ML}/figures'

mlflow.set_experiment('/Users/jorgemejia251@outlook.com/semi2_ml_mortality')


def _s3():
    """Cliente boto3 con creds del scope semis2_scope (mismo patron que s3.py)."""
    return boto3.client(
        's3', region_name='us-east-1',
        aws_access_key_id=dbutils.secrets.get('semis2_scope', 'AWS_ACCESS_KEY_ID'),
        aws_secret_access_key=dbutils.secrets.get('semis2_scope', 'AWS_SECRET_ACCESS_KEY'))


def _upload(local, key):
    try:
        _s3().upload_file(local, BUCKET, key)
        print(f'  s3 -> s3://{BUCKET}/{key}')
    except Exception as e:
        print(f'[WARN] S3 no disponible: {e} (queda en {local})')


def _export_pred(pdf, table):
    """Salva pred_* en Unity Catalog + la sube a S3 como parquet y csv (pandas->boto3)."""
    sdf = spark.createDataFrame(pdf)
    sdf.write.mode('overwrite').saveAsTable(f'{CAT}.{table}')
    os.makedirs('/tmp/ml_export', exist_ok=True)
    for fmt in ('parquet', 'csv'):
        local = f'/tmp/ml_export/{table}.{fmt}'
        if fmt == 'parquet':
            pdf.to_parquet(local, index=False)
        else:
            pdf.to_csv(local, index=False)
        _upload(local, f'{S3_PRED}/{table}.{fmt}')
    print(f'pred -> {CAT}.{table}  +  s3://{BUCKET}/{S3_PRED}/{table}.parquet')


def _save_model(modelo, name):
    os.makedirs('/tmp/models', exist_ok=True)
    local = f'/tmp/models/{name}.joblib'
    joblib.dump(modelo, local)
    _upload(local, f'{S3_MODELS}/{name}/model.joblib')


def _save_fig(fig, name):
    os.makedirs('/tmp/figures', exist_ok=True)
    local = f'/tmp/figures/{name}.png'
    fig.savefig(local, dpi=130, bbox_inches='tight')
    _upload(local, f'{S3_FIG}/{name}.png')

## Modelo A — Exceso de mortalidad (Regresión Lineal)
Construye una **línea base** de cuántas defunciones habrían ocurrido sin pandemia (entrenada con los años previos a 2020, incluyendo el patrón estacional de cada mes) y la proyecta sobre 2020–2024. La diferencia entre lo observado y lo esperado es el **exceso**.

In [ ]:
abt = spark.table(f'{CAT}.abt_ine').toPandas()
serie = abt.groupby(['anio','mes','es_pre_covid'], as_index=False)['defunciones'].sum()
feat = lambda d: pd.get_dummies(d[['anio','mes']], columns=['mes'])
pre  = serie[serie.es_pre_covid == True]
Xtr, ytr = feat(pre), pre['defunciones']
Xall = feat(serie).reindex(columns=Xtr.columns, fill_value=0)

with mlflow.start_run(run_name='modelo_A_exceso'):
    m = LinearRegression().fit(Xtr, ytr)
    r2_a = r2_score(ytr, m.predict(Xtr))
    serie['esperado'] = np.round(m.predict(Xall), 1)
    serie['exceso']   = serie['defunciones'] - serie['esperado']
    pred_exceso = serie.rename(columns={'defunciones':'observado'})
    exceso_total = float(pred_exceso.query('not es_pre_covid').exceso.sum())
    mlflow.log_params({'train_window':'2015-2019','features':'anio+mes_onehot','model':'LinearRegression'})
    mlflow.log_metrics({'r2_pre': r2_a, 'exceso_total_2020_2024': exceso_total})
    try:
        mlflow.sklearn.log_model(m, 'model', registered_model_name=f'{CAT}.exceso_lr')
    except Exception as e:
        print('[mlflow] registro UC omitido:', e)
        mlflow.sklearn.log_model(m, 'model')
    _save_model(m, 'exceso_lr')
    print('R2 pre =', round(r2_a, 3))
    print('exceso total 2020-2024 =', round(exceso_total))

_export_pred(pred_exceso, 'pred_exceso')
display(pred_exceso)

## Modelo B — Drivers de mortalidad (Random Forest)
Predice cuántas defunciones ocurren para cada combinación de departamento, causa, sexo y grupo de edad, y expone **qué factores pesan más** en esa predicción.

In [ ]:
cats = ['departamento','causa_capitulo','sexo','grupo_edad']
X = pd.get_dummies(abt[cats + ['anio','mes']], columns=cats)
y = abt['defunciones']
Xtr,Xte,ytr,yte = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name='modelo_B_drivers'):
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    r2_b = r2_score(yte, rf.predict(Xte))
    mlflow.log_params({'model':'RandomForestRegressor','n_estimators':200,
                       'features': ','.join(cats) + ',anio,mes', 'test_size': 0.2})
    mlflow.log_metrics({'r2_test': r2_b})
    try:
        mlflow.sklearn.log_model(rf, 'model', registered_model_name=f'{CAT}.rf_mortality')
    except Exception as e:
        print('[mlflow] registro UC omitido:', e)
        mlflow.sklearn.log_model(rf, 'model')
    _save_model(rf, 'rf_mortality')
    print('R2 test =', round(r2_b, 3))

imp = (pd.Series(rf.feature_importances_, index=X.columns)
         .sort_values(ascending=False).head(20)
         .rename_axis('factor').reset_index(name='importancia'))
_export_pred(imp, 'pred_rf_importancias')
display(imp)

## Modelo C — Segmentación territorial (K-means)
Agrupa los departamentos según su **perfil de causas de muerte** (qué porcentaje corresponde a cada capítulo CIE-10), antes y después de la pandemia, y marca **cuáles cambiaron de grupo**.

In [ ]:
pre_df  = spark.table(f'{CAT}.abt_ine_wide_pre').toPandas().assign(periodo='pre')
post_df = spark.table(f'{CAT}.abt_ine_wide_post').toPandas().assign(periodo='post')

feats = sorted(set(c for c in pre_df.columns if c.startswith('pct_'))
               & set(c for c in post_df.columns if c.startswith('pct_')))
both = pd.concat([pre_df, post_df], ignore_index=True)
Xs = StandardScaler().fit_transform(both[feats].fillna(0))

with mlflow.start_run(run_name='modelo_C_segmentacion'):
    km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(Xs)
    sil = silhouette_score(Xs, km.labels_)
    both['cluster'] = km.labels_
    mlflow.log_params({'model':'KMeans','k':4,'features':'pct_capitulos_cie10',
                       'periodo':'pre+post_juntos'})
    mlflow.log_metrics({'silhouette': sil})
    try:
        mlflow.sklearn.log_model(km, 'model', registered_model_name=f'{CAT}.kmeans_segments')
    except Exception as e:
        print('[mlflow] registro UC omitido:', e)
        mlflow.sklearn.log_model(km, 'model')
    _save_model(km, 'kmeans_segments')
    print('silhouette (pre+post juntos) =', round(sil, 3))

seg = (both.pivot_table(index='departamento', columns='periodo',
                        values='cluster', aggfunc='first').reset_index())
seg.columns.name = None
seg = seg.rename(columns={'pre': 'cluster_pre', 'post': 'cluster_post'})
seg['cambio'] = seg['cluster_pre'] != seg['cluster_post']
_export_pred(seg, 'pred_clusters_departamento')
print('departamentos que cambiaron de patron:', int(seg.cambio.sum()), 'de', len(seg))
display(seg.sort_values('cambio', ascending=False))

## Verificación — tablas de predicción
Confirmamos que cada modelo dejó su tabla escrita y lista para consultar.

In [ ]:
for t in ['pred_exceso','pred_rf_importancias','pred_clusters_departamento']:
    print(t, '->', spark.table(f'{CAT}.{t}').count(), 'filas')
display(spark.table(f'{CAT}.pred_exceso'))

# Gráficas — diagnóstico de los modelos

Antes de llevar los resultados a los dashboards, los dibujamos acá para comprobar que la
historia que cuentan tiene sentido. Cada modelo responde una pregunta distinta:

- **Modelo A** → ¿dónde y cuándo aparece el exceso de mortalidad?
- **Modelo B** → ¿qué factores pesan más y qué tan bien predice el modelo?
- **Modelo C** → ¿cómo se redistribuyen los departamentos entre clusters antes y después?
- **Modelo D** → ¿cuánto cambia el pronóstico según el supuesto que elijamos?

Una misma paleta y estilo para todos, para que se lean como parte de un solo reporte.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

# --- Paleta pastel unificada (P = rellenos, D = acentos/texto) ---
P = dict(rojo='#F5B7B1', azul='#AED6F1', verde='#A9DFBF', naranja='#FAD7A0',
         morado='#D7BDE2', amarillo='#F9E79F', gris='#D5DBDB', teal='#A2D9CE',
         rosa='#FADBD8', cielo='#AED6F1')
D = dict(rojo='#C0392B', azul='#2471A3', verde='#1E8449', naranja='#B9770E',
         morado='#7D3C98', gris='#566573', teal='#117A65')
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#FBFCFC',
    'axes.edgecolor': '#D5DBDB', 'axes.grid': True, 'grid.color': '#EAECEE',
    'grid.alpha': .55, 'axes.titlesize': 12.5, 'axes.titleweight': 'bold',
    'axes.titlecolor': '#2C3E50', 'axes.labelsize': 10, 'axes.labelcolor': '#34495E',
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'xtick.color': '#566573', 'ytick.color': '#566573',
    'legend.frameon': False, 'figure.dpi': 110,
})

# --- Modelo A: exceso de mortalidad ---
ex = spark.table(f'{CAT}.pred_exceso').toPandas()
g = (ex.groupby('anio')
       .agg(observado=('observado', 'sum'), esperado=('esperado', 'sum'),
            exceso=('exceso', 'sum')).reset_index())

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, hspace=.35, wspace=.28)
a1 = fig.add_subplot(gs[0, :2])   # serie observado vs esperado
a2 = fig.add_subplot(gs[0, 2])     # exceso por año (barras)
a3 = fig.add_subplot(gs[1, :])     # heatmap mensual año × mes

# (a1) Observado vs esperado: la brecha es el exceso. Área suave para que se vea el
# "sobre-crecimiento" desde 2020 sin que compita con las líneas.
a1.plot(g.anio, g.observado, 'o-', color=D['rojo'], lw=2.4, ms=6, label='Observado', zorder=3)
a1.plot(g.anio, g.esperado, 's--', color=D['azul'], lw=2, ms=5, label='Esperado (sin COVID)', zorder=3)
a1.fill_between(g.anio, g.esperado, g.observado, where=g.observado >= g.esperado,
                color=P['rojo'], alpha=.45, label='Exceso', zorder=1)
a1.fill_between(g.anio, g.observado, g.esperado, where=g.observado < g.esperado,
                color=P['verde'], alpha=.45, label='Déficit', zorder=1)
a1.axvline(2019.5, color=D['gris'], ls=':', lw=1.2)
a1.text(2019.6, g.observado.max(), '  inicio COVID',
        color=D['gris'], fontsize=9, va='top')
peak = g.loc[g.exceso.idxmax()]
a1.annotate(f'pico exceso\n{int(peak.exceso):,}', (peak.anio, peak.observado),
            textcoords='offset points', xytext=(-10, 22), color=D['rojo'], fontsize=9,
            ha='center', arrowprops=dict(arrowstyle='->', color=D['rojo'], lw=1.2))
a1.set_title('Serie anual: defunciones observadas vs esperadas')
a1.set_xlabel('Año'); a1.set_ylabel('Defunciones'); a1.legend(loc='upper left', ncol=2)
a1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{int(v/1000)}k'))

# (a2) Exceso por año post-COVID: barra arriba/abajo del cero, color según signo.
post = g[g.anio >= 2020].copy()
cols = [P['rojo'] if v >= 0 else P['verde'] for v in post.exceso]
bars = a2.bar(post.anio, post.exceso, color=cols, edgecolor='white', width=.7)
a2.bar_label(bars, fmt='%.0f', fontsize=8, color=D['gris'])
a2.axhline(0, color=D['gris'], lw=.8)
a2.set_title('Exceso anual 2020–2024')
a2.set_xlabel('Año'); a2.set_ylabel('Exceso (defunciones)')
a2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{int(v/1000)}k'))

# (a3) Heatmap mensual: año × mes del exceso. Muestra la estacionalidad y dónde se
# concentra el exceso (meses fríos suelen tener más mortalidad, COVID rompe el patrón).
hm = ex.pivot_table(index='anio', columns='mes', values='exceso', aggfunc='sum').sort_index()
im = a3.imshow(hm.values, cmap='RdBu_r', aspect='auto',
               vmin=-abs(hm.values).max(), vmax=abs(hm.values).max())
a3.set_xticks(range(12)); a3.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun',
                                              'Jul','Ago','Sep','Oct','Nov','Dic'], fontsize=9)
a3.set_yticks(range(len(hm))); a3.set_yticklabels(hm.index.astype(int))
a3.set_title('Exceso mensual (heatmap) — rojo = más muertes de las esperadas')
a3.set_xlabel('Mes'); a3.set_ylabel('Año')
# anotar los valores grandes para que el heatmap se lea sin la barra de color
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        v = hm.values[i, j]
        if abs(v) > hm.values.std():
            a3.text(j, i, f'{int(v):,}', ha='center', va='center', fontsize=7.5,
                    color='white' if abs(v) > hm.values.std()*1.8 else D['gris'])
cb = fig.colorbar(im, ax=a3, fraction=.025, pad=.01)
cb.set_label('Exceso (defunciones)', fontsize=9)
fig.suptitle('Modelo A — Exceso de mortalidad GT (INE)', fontsize=14, weight='bold', color='#2C3E50', y=.97)
_save_fig(fig, '01_exceso')
plt.show()

### Lectura — Modelo A (exceso)

- La línea azul es lo que **habría pasado** sin pandemia: la tendencia y el patrón mensual
  de los años previos, proyectados hacia adelante. La brecha roja por encima es el exceso.
- El exceso no fue parejo: el **pico de 2021** (segunda ola) concentra la mayor parte,
  2020 y 2022 también quedan altos, y hacia 2023–2024 la brecha empieza a cerrarse.
- El **heatmap mensual** muestra algo que la serie anual esconde: los meses fríos
  (nov–feb) siempre cargan más defunciones, y el exceso COVID se concentró en ciertos
  meses, no en todos por igual. Rojo intenso = más muertes de las esperadas, azul = menos.

In [ ]:
# --- Modelo B: factores que más explican la mortalidad + ajuste real vs predicho ---
imp = spark.table(f'{CAT}.pred_rf_importancias').toPandas().head(12).iloc[::-1]

fig, (b1, b2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [1.25, 1]})

# (b1) Importancia: gradiente pastel verde→azul según ranking, para que el ojo vaya
# primero al que más pesa sin que el color grite.
vals = imp.importancia.values
norm = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('imp', [P['verde'], P['azul']])
bar_cols = [cmap(n) for n in norm]
b1.barh(imp.factor, imp.importancia, color=bar_cols, edgecolor='white', height=.72)
for y, (f, v) in enumerate(zip(imp.factor, imp.importancia)):
    b1.text(v + vals.max()*0.01, y, f'  {v:.3f}', va='center', fontsize=8.5, color=D['gris'])
b1.set_title('Top 12 factores que más explican las defunciones')
b1.set_xlabel('Importancia (ganancia de información)')
b1.margins(x=.18)

# (b2) Dispersión real vs predicho sobre el set de test. La diagonal y=x es el
# "modelo perfecto"; la nube pegada a ella = buen ajuste. R² va como anotación.
# Reconstruimos split y modelo rápido (ya entrenado arriba) para tener y_hat de test.
cats = ['departamento','causa_capitulo','sexo','grupo_edad']
Xb = pd.get_dummies(abt[cats + ['anio','mes']], columns=cats)
yb = abt['defunciones']
Xtr,Xte,ytr,yte = train_test_split(Xb, yb, test_size=0.2, random_state=42)
rfd = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr, ytr)
yp = rfd.predict(Xte)
r2_b = r2_score(yte, yp)
mx = max(yte.max(), yp.max())
b2.plot([0, mx], [0, mx], '--', color=D['gris'], lw=1.3, label='predicción ideal (y=x)')
b2.scatter(yte, yp, s=14, color=P['morado'], alpha=.55, edgecolor='none', label='observaciones test')
b2.set_title(f'Real vs predicho (Random Forest)')
b2.set_xlabel('Defunciones reales'); b2.set_ylabel('Defunciones predichas')
b2.text(.05, .92, f'R² test = {r2_b:.3f}', transform=b2.transAxes,
        fontsize=11, weight='bold', color=D['morado'],
        bbox=dict(boxstyle='round,pad=.4', fc=P['morado'], ec='none'))
b2.legend(loc='lower right')
b2.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{int(v/1000)}k'))
b2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{int(v/1000)}k'))
fig.suptitle('Modelo B — Drivers de mortalidad (Random Forest)', fontsize=14, weight='bold', color='#2C3E50', y=1.02)
plt.tight_layout(); _save_fig(fig, '02_drivers_rf')
plt.show()

### Lectura — Modelo B (drivers)

- El factor que más pesa suele ser la **causa de muerte** (capítulo CIE-10), seguido del
  **departamento** y el **grupo de edad**. El año y el mes pesan menos porque la causa
  ya captura gran parte de la variación.
- El gráfico **real vs predicho** debería quedar pegado a la diagonal. Un R² alto (≈0.9)
  significa que el modelo reproduce bien lo observado entre 2015 y 2024 —no que prediga
  el futuro. Si la nube se abre en las colas (pocos casos con muchas defunciones), es
  justo donde el modelo se equivoca más.

In [ ]:
# --- Modelo C: segmentación territorial pre vs post ---
seg = spark.table(f'{CAT}.pred_clusters_departamento').toPandas()
ks = sorted(set(seg.cluster_pre) | set(seg.cluster_post))
pre_n  = [int((seg.cluster_pre == k).sum()) for k in ks]
post_n = [int((seg.cluster_post == k).sum()) for k in ks]
x = np.arange(len(ks)); w = 0.38

fig = plt.figure(figsize=(16, 6))
gs = fig.add_gridspec(1, 3, wspace=.32)
c1 = fig.add_subplot(gs[0, 0])   # tamaño de clusters pre vs post
c2 = fig.add_subplot(gs[0, 1])   # donut post-COVID
c3 = fig.add_subplot(gs[0, 2])   # dot-chart de cambios

# Paleta fija por cluster (mismos colores en los 3 paneles para que sean comparables)
ck = [P['azul'], P['rojo'], P['verde'], P['naranja']][:len(ks)]
ckd = [D['azul'], D['rojo'], D['verde'], D['naranja']][:len(ks)]

# (c1) Barras agrupadas: cuántos departamentos caen en cada cluster, antes y después.
# Pre = color sólido, post = mismo color con hatch y algo de transparencia (distingue
# sin meter una segunda paleta que rompería la comparación con el donut y el dot-chart).
c1.bar(x - w/2, pre_n, w, label='Pre-COVID', color=ck, edgecolor='white')
c1.bar(x + w/2, post_n, w, label='Post-COVID', color=ck,
       edgecolor='white', alpha=.5, hatch='///')
for xi, pv, po in zip(x, pre_n, post_n):
    c1.text(xi - w/2, pv + .15, pv, ha='center', fontsize=9, color=D['gris'])
    c1.text(xi + w/2, po + .15, po, ha='center', fontsize=9, color=D['gris'])
c1.set_xticks(x); c1.set_xticklabels([f'Cluster {k}' for k in ks])
c1.set_title('Tamaño de cada cluster (pre vs post)')
c1.set_ylabel('N° departamentos'); c1.legend()

# (c2) Donut: cómo queda repartido el país post-COVID. Más claro que la barra para
# transmitir "proporción".
wedges, _, autotxt = c2.pie(post_n, labels=[f'C{k}' for k in ks], colors=ck,
                            autopct='%1.0f%%', startangle=90,
                            wedgeprops=dict(width=.42, edgecolor='white'),
                            textprops=dict(color=D['gris'], fontsize=9))
for at in autotxt:
    at.set_color('white'); at.set_weight('bold')
c2.set_title('Composición post-COVID')
c2.text(0, 0, f'{len(seg)}\ndepart.', ha='center', va='center', fontsize=11, color=D['gris'])

# (c3) Dot-chart: para cada departamento, un punto en su cluster_pre y otro en cluster_post
# unidos por una línea. Las líneas que cruzan = cambios de patrón. Mucho más informativo
# que la barra "sí/no cambió" porque muestra DE DÓNDE A DÓNDE se movió.
order = seg.sort_values(['cambio','departamento'], ascending=[False, True]).reset_index(drop=True)
ypos = np.arange(len(order))[::-1]
for i, r in order.iterrows():
    yi = ypos[i]
    c3.plot([r.cluster_pre, r.cluster_post], [yi, yi],
            color=D['naranja'] if r.cambio else P['gris'],
            lw=2 if r.cambio else 1, alpha=.85 if r.cambio else .5, zorder=1)
    c3.scatter(r.cluster_pre, yi, color=ck[r.cluster_pre], s=70, zorder=3, edgecolor='white')
    c3.scatter(r.cluster_post, yi, color=ck[r.cluster_post], s=70, zorder=3, edgecolor='white', marker='D')
c3.set_yticks(ypos); c3.set_yticklabels(order.departamento, fontsize=8.5)
c3.set_xticks(ks); c3.set_xticklabels([f'C{k}' for k in ks])
c3.set_title(f'Trayectoria pre → post  (cambiaron {int(order.cambio.sum())} de {len(order)})')
c3.set_xlabel('Cluster')
# leyenda manual para distinguir punto pre (círculo) de post (rombo)
from matplotlib.lines import Line2D
c3.legend(handles=[Line2D([0],[0], marker='o', color='none', markerfacecolor=D['gris'], markersize=9, label='pre'),
                   Line2D([0],[0], marker='D', color='none', markerfacecolor=D['gris'], markersize=8, label='post'),
                   Line2D([0],[0], color=D['naranja'], lw=2, label='cambió de cluster')],
          loc='lower right', fontsize=8)
fig.suptitle('Modelo C — Segmentación territorial (K-means, pre vs post COVID)', fontsize=14, weight='bold', color='#2C3E50', y=1.03)
plt.tight_layout(); _save_fig(fig, '03_segmentacion')
plt.show()

### Lectura — Modelo C (segmentación)

- Las barras y el donut muestran si el **tamaño** de cada cluster cambia antes y después:
  si un grupo crece mucho, varios departamentos se "corrieron" de perfil.
- El **dot-chart** es el más revelador: cada línea es un departamento, con un punto en su
  cluster anterior y otro en el actual. Las líneas naranjas que cruzan de un lado a otro
  marcan un cambio de patrón; las que se quedan cortas indican que el perfil de causas
  se mantuvo.
- Que **pocos** departamentos cambien de cluster dice que la estructura territorial de
  causas es bastante estable, y que el COVID la alteró solo en casos puntuales.

# Modelo D — Pronóstico de defunciones 2025–2027

Proyecta cuántas defunciones podrían ocurrir en los próximos tres años. **Es una extrapolación**
(años para los que todavía no hay datos), así que el resultado depende del supuesto de partida.
Elegimos el escenario **"nueva normalidad"** (entrena solo con 2022–2024, ya post-pandemia)
porque es el más honesto: no asume que el COVID se borra (eso sería optimista) ni arrastra
el pico anómalo de 2021 (eso inflaría la tendencia). Se presenta como proyección con salvedad,
no como certeza.

In [ ]:
serie_m = abt.groupby(['anio', 'mes'], as_index=False)['defunciones'].sum()
featm = lambda d: pd.get_dummies(d[['anio', 'mes']], columns=['mes'])

def forecast(train_df, years=(2025, 2026, 2027)):
    Xtr = featm(train_df)
    mod = LinearRegression().fit(Xtr, train_df['defunciones'])
    fut = pd.DataFrame([(a, m) for a in years for m in range(1, 13)],
                       columns=['anio', 'mes'])
    Xf = featm(fut).reindex(columns=Xtr.columns, fill_value=0)
    fut['defunciones_pred'] = np.round(mod.predict(Xf), 0)
    sigma = int(round((train_df['defunciones'] - mod.predict(Xtr)).std(), -2))
    fut['lim_inf'] = fut['defunciones_pred'] - sigma
    fut['lim_sup'] = fut['defunciones_pred'] + sigma
    return fut, mod

with mlflow.start_run(run_name='modelo_D_forecast'):
    fc, mod_d = forecast(serie_m[serie_m.anio >= 2022])
    fc['escenario'] = 'nueva_normalidad_2022_2024'
    r2_d = r2_score(serie_m[serie_m.anio >= 2022]['defunciones'],
                    mod_d.predict(featm(serie_m[serie_m.anio >= 2022])))
    mlflow.log_params({'model':'LinearRegression','train_window':'2022-2024',
                       'horizon':'2025-2027','escenario':'nueva_normalidad'})
    mlflow.log_metrics({'r2_ajuste_2022_2024': r2_d})
    try:
        mlflow.sklearn.log_model(mod_d, 'model', registered_model_name=f'{CAT}.forecast_lr')
    except Exception as e:
        print('[mlflow] registro UC omitido:', e)
        mlflow.sklearn.log_model(mod_d, 'model')
    _save_model(mod_d, 'forecast_lr')
    print(f'R2 ajuste 2022-2024 = {r2_d:.3f}')

_export_pred(fc, 'pred_forecast')
print('Pronostico anual (punto + intervalo):')
res = (fc.groupby('anio')
         .agg(pred=('defunciones_pred', 'sum'),
              lim_inf=('lim_inf', 'sum'),
              lim_sup=('lim_sup', 'sum'))
         .astype(int))
print(res.to_string())
display(fc)

In [ ]:
# Comparación de los 3 escenarios + desglose mensual del pronóstico elegido.
# Panel izq: la "historia" — cuánto cambia el 2027 según el supuesto. Panel der:
# cómo se reparte el pronóstico mes a mes (estacionalidad que la línea anual esconde).
anual = abt.groupby('anio', as_index=False)['defunciones'].sum()

def annual_proj(train, years=(2025, 2026, 2027)):
    mod = LinearRegression().fit(train[['anio']], train['defunciones'])
    fut = pd.DataFrame({'anio': list(years)})
    return fut['anio'].tolist(), np.round(mod.predict(fut)).astype(int).tolist()

# Esc.1: 2015-2019 → "el COVID fue temporal, volvemos a la normalidad" (optimista).
# Esc.2: 2015-2024 → el pico 2021 infla la pendiente, por eso sale más alto (engaña).
# Esc.3: 2022-2024 → la "nueva normalidad"; es el que de verdad usamos.
yrs, s1 = annual_proj(anual[anual.anio <= 2019])
_,   s2 = annual_proj(anual)
_,   s3 = annual_proj(anual[anual.anio >= 2022])

# Para que el pronóstico se lea como continuación del histórico (y no como puntos
# flotando a la derecha), cada escenario arranca en el último año real (2024).
last_anio = int(anual.anio.max())
last_val  = int(anual.defunciones.iloc[-1])
fx  = [last_anio] + yrs
fs1 = [last_val] + s1
fs2 = [last_val] + s2
fs3 = [last_val] + s3

fig, (ax, hm_ax) = plt.subplots(1, 2, figsize=(17, 6.5),
                                gridspec_kw={'width_ratios': [1.4, 1]})

# Histórico en negro, escenarios en pastel. El esc.3 (usado) lleva más grosor y
# marcador diamante para que el ojo lo identifique como la proyección oficial.
ax.plot(anual.anio, anual.defunciones, 'o-', color='#2C3E50', lw=2.4, ms=5, label='Histórico (real)')
ax.plot(fx, fs1, 's--', color=D['verde'], lw=1.8, label='Esc.1 pre-COVID (optimista)')
ax.plot(fx, fs2, '^--', color=D['gris'], lw=1.8, label='Esc.2 serie completa (inflado)')
ax.plot(fx, fs3, 'D-', color=D['rojo'], lw=2.8, ms=6, label='Esc.3 nueva normalidad (usado)')

# Banda del intervalo (±1σ del cell anterior) sobre el escenario elegido, agregada
# a anual desde pred_forecast (lim_inf/lim_sup mensual). Así el punto no va "pelado".
band = (fc.groupby('anio')
          .agg(lo=('lim_inf', 'sum'), hi=('lim_sup', 'sum')).astype(int))
bx = [last_anio] + band.index.tolist()
ax.fill_between(bx, [last_val] + band.lo.tolist(),
                [last_val] + band.hi.tolist(),
                color=P['rojo'], alpha=.35, label='Esc.3 intervalo (±1σ)')

# Anotar el valor 2027 de cada escenario al lado del punto — el número tiene que
# verse sin tener que leer la leyenda ni el eje.
ax.annotate(f'{s1[-1]:,}', (yrs[-1], s1[-1]), textcoords='offset points',
            xytext=(8, 4), color=D['verde'], fontsize=9)
ax.annotate(f'{s2[-1]:,}', (yrs[-1], s2[-1]), textcoords='offset points',
            xytext=(8, -12), color=D['gris'], fontsize=9)
ax.annotate(f'{s3[-1]:,}', (yrs[-1], s3[-1]), textcoords='offset points',
            xytext=(8, 6), color=D['rojo'], fontsize=10.5, weight='bold')
# Delta de los ~20k que separan al optimista del inflado: la cifra que se defiende.
ax.annotate('', xy=(yrs[-1]+.15, s2[-1]), xytext=(yrs[-1]+.15, s1[-1]),
            arrowprops=dict(arrowstyle='<->', color=D['gris'], lw=1.1))
ax.text(yrs[-1]+.25, (s1[-1]+s2[-1])/2, f'Δ≈{s2[-1]-s1[-1]:,}\nsegún supuesto',
        color=D['gris'], fontsize=8.5, va='center')

ax.axvspan(2024.5, 2027.5, alpha=.08, color=P['gris'])
ax.text(2026, last_val * .82, 'Pronóstico', ha='center', color=D['gris'], fontsize=10)
ax.set_xlim(2014, 2028.2)
ax.set_title('Defunciones GT 2025–2027 — 3 escenarios')
ax.set_xlabel('Año'); ax.set_ylabel('Defunciones')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{int(v/1000)}k'))
ax.legend(loc='upper left', framealpha=.9)

# Heatmap mensual del pronóstico (esc.3): año × mes. Muestra la estacionalidad que
# la línea anual oculta — meses fríos con más defunciones, verano con menos.
hm = fc.pivot_table(index='anio', columns='mes', values='defunciones_pred', aggfunc='sum').sort_index()
im = hm_ax.imshow(hm.values, cmap='Reds', aspect='auto')
hm_ax.set_xticks(range(12)); hm_ax.set_xticklabels(['E','F','M','A','M','J','J','A','S','O','N','D'], fontsize=9)
hm_ax.set_yticks(range(len(hm))); hm_ax.set_yticklabels(hm.index.astype(int))
hm_ax.set_title('Pronóstico mensual (Esc.3)')
hm_ax.set_xlabel('Mes'); hm_ax.set_ylabel('Año')
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        hm_ax.text(j, i, f'{int(hm.values[i,j])}', ha='center', va='center',
                   fontsize=7.5, color='white' if hm.values[i,j] > hm.values.mean() else D['gris'])
cb = fig.colorbar(im, ax=hm_ax, fraction=.04, pad=.02)
cb.set_label('Defunciones', fontsize=9)
fig.suptitle('Modelo D — Pronóstico de defunciones GT 2025–2027', fontsize=14, weight='bold', color='#2C3E50', y=1.02)
plt.tight_layout(); _save_fig(fig, '04_forecast')
plt.show()

### Lectura — Modelo D (pronóstico)

- Los tres escenarios se separan por ~20 mil muertes en 2027: esa es la cifra que hay
  que justificar. El **Esc.1** (verde) es optimista, asume que el COVID se borra; el
  **Esc.2** (gris) queda inflado por el pico de 2021; el **Esc.3** (rojo, el que usamos)
  entrena solo con 2022–2024 y refleja la "nueva normalidad".
- La **banda ±1σ** sobre el Esc.3 es lo honesto: decir "≈105k ± X" en lugar de un punto
  falso de precisión. No es un intervalo de confianza formal (para eso haría falta un
  modelo de serie de tiempo más sofisticado), pero evita sobre-vender el número.
- El **heatmap mensual** recuerda que el total anual esconde la estacionalidad: los meses
  fríos concentran más defunciones. Como es una extrapolación a 3 años fuera de los datos,
  conviene presentarlo siempre con la salvedad de que la tendencia pasada se mantiene recta.